# Task 4 - Optimizacion de Parametros

Este notebook documenta la busqueda sistematica de hiperparametros para los modelos mas prometedores de Task 3. La metrica principal sigue siendo `recall_condition_1`, porque en este contexto medico el falso negativo es el error mas critico: clasificar como sano a un paciente que realmente presenta la condicion.

## Modelos seleccionados

A partir de Task 3 se optimizan tres modelos: Regresion Logistica, SVM RBF y KNN. Regresion Logistica fue la opcion mas solida en validacion cruzada, SVM RBF redujo falsos negativos en el test inicial y KNN queda como tercer candidato competitivo para contrastar una frontera no parametrica.

In [1]:
from pathlib import Path
import sys

import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd / "mundial_project" / "src").exists():
    PROJECT_ROOT = cwd / "mundial_project"
elif (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise RuntimeError(f"No se encontro la carpeta src desde {cwd}")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

TASK3_DIR = PROJECT_ROOT / "outputs" / "task3"
TASK4_DIR = PROJECT_ROOT / "outputs" / "task4"
PROJECT_ROOT

WindowsPath('E:/University/4to/ML/mundial_project')

## Resultados base de Task 3

La comparacion previa se usa como punto de partida. La optimizacion no se decide por accuracy, sino por recall de `condition=1`, falsos negativos y coste medico.

In [2]:
task3_summary = pd.read_csv(TASK3_DIR / "comparacion_modelos_resumen.csv")
task3_summary

,modelo,cv_recall_condition_1_mean,cv_recall_condition_1_std,cv_roc_auc_mean,cv_medical_cost_mean,cv_false_negatives_mean,test_recall_condition_1,test_roc_auc,test_false_negatives,test_false_positives,test_medical_cost
0,Regresion Logistica,0.813521,0.085339,0.915978,0.493505,4.066667,0.821429,0.896205,5.0,6.0,0.516667
1,SVM RBF,0.806638,0.079749,0.911232,0.517021,4.213333,0.857143,0.881696,4.0,8.0,0.466667
2,KNN,0.785281,0.076233,0.880634,0.572512,4.680000,0.821429,0.848772,5.0,6.0,0.516667
3,Naive Bayes,0.728167,0.149570,0.883075,0.698443,5.926667,0.821429,0.838170,5.0,5.0,0.500000
4,Arbol de Decision,0.719293,0.095358,0.825273,0.739938,6.113333,0.785714,0.824777,6.0,10.0,0.666667
5,Red Neuronal MLP,0.572020,0.154867,0.805088,1.055083,9.326667,0.250000,0.714286,21.0,4.0,1.816667


## Ejecucion de experimentos

Los scripts independientes se pueden ejecutar desde terminal. Para evitar relanzar busquedas largas accidentalmente al abrir el notebook, la celda siguiente solo corre los experimentos si `RUN_EXPERIMENTS = True`.

In [3]:
RUN_EXPERIMENTS = False

if RUN_EXPERIMENTS:
    from task4_logistic_regression_optimization import run as run_logistic
    from task4_svm_optimization import run as run_svm
    from task4_knn_optimization import run as run_knn
    import task4_comparacion_optimizacion

    run_logistic()
    run_svm()
    run_knn()
    comparison = task4_comparacion_optimizacion.build_comparison(TASK4_DIR, TASK3_DIR)
    feature_ranking = task4_comparacion_optimizacion.build_global_feature_ranking(TASK4_DIR)
    task4_comparacion_optimizacion.write_report(
        comparison,
        feature_ranking,
        PROJECT_ROOT / "reports" / "task4_optimizacion_parametros.md",
    )

## Comparacion de modelos optimizados

La tabla consolidada ordena por recall medio en validacion cruzada, luego por menor coste medico. El test final se conserva como verificacion externa y ayuda a detectar configuraciones fragiles.

In [4]:
ranking = pd.read_csv(TASK4_DIR / "ranking_modelos_optimizados.csv")
ranking

,modelo,cv_recall_condition_1_mean,cv_medical_cost_mean,cv_roc_auc_mean,test_recall_condition_1,test_false_negatives,test_false_positives,test_medical_cost,delta_cv_recall,delta_cv_medical_cost
0,SVM RBF,1.000000,0.540071,0.846331,0.000000,28.0,0.0,2.333333,0.193362,0.023050
1,Regresion Logistica,0.847965,0.421667,0.922432,0.821429,5.0,8.0,0.550000,0.034444,-0.071838
2,KNN,0.795455,0.537172,0.909738,0.821429,5.0,5.0,0.500000,0.010173,-0.035340


In [5]:
comparison = pd.read_csv(TASK4_DIR / "comparacion_optimizacion.csv")
comparison[["modelo", "best_params", "cv_recall_condition_1_mean", "test_recall_condition_1", "test_false_negatives", "test_false_positives", "test_medical_cost"]]

,modelo,best_params,cv_recall_condition_1_mean,test_recall_condition_1,test_false_negatives,test_false_positives,test_medical_cost
0,SVM RBF,"{'model__C': 0.1, 'model__class_weight': 'bala...",1.000000,0.000000,28.0,0.0,2.333333
1,Regresion Logistica,"{'model__C': 1.0, 'model__class_weight': 'bala...",0.847965,0.821429,5.0,8.0,0.550000
2,KNN,"{'model__n_neighbors': 7, 'model__p': 1, 'mode...",0.795455,0.821429,5.0,5.0,0.500000


## Interpretacion medica

La seleccion por recall puede favorecer modelos que detectan mas positivos en validacion, pero tambien puede elevar falsos positivos o mostrar fragilidad en test. Por eso el analisis final revisa juntos `recall_condition_1`, falsos negativos, falsos positivos, coste medico y ROC-AUC.

In [6]:
best_cv = comparison.sort_values(
    ["cv_recall_condition_1_mean", "cv_medical_cost_mean", "test_false_negatives"],
    ascending=[False, True, True],
).iloc[0]
best_test_cost = comparison.sort_values("test_medical_cost", ascending=True).iloc[0]

pd.DataFrame([
    {
        "criterio": "Mejor por validacion cruzada",
        "modelo": best_cv["modelo"],
        "recall_cv": best_cv["cv_recall_condition_1_mean"],
        "coste_test": best_cv["test_medical_cost"],
        "fn_test": best_cv["test_false_negatives"],
    },
    {
        "criterio": "Menor coste en test",
        "modelo": best_test_cost["modelo"],
        "recall_cv": best_test_cost["cv_recall_condition_1_mean"],
        "coste_test": best_test_cost["test_medical_cost"],
        "fn_test": best_test_cost["test_false_negatives"],
    },
])

,criterio,modelo,recall_cv,coste_test,fn_test
0,Mejor por validacion cruzada,SVM RBF,1.000000,2.333333,28.0
1,Menor coste en test,KNN,0.795455,0.500000,5.0


## Variables mas influyentes

La influencia se calcula con `permutation_importance` sobre las variables originales del dataset y usando recall como scoring. Una importancia mayor significa que permutar esa variable reduce mas la capacidad del modelo para detectar `condition=1`.

In [7]:
feature_ranking = pd.read_csv(TASK4_DIR / "variables_influyentes_global.csv")
feature_ranking

,feature,importance_normalized_mean,importance_mean_avg,models_with_positive_importance
0,feature_8,0.666667,0.042857,2
1,feature_12,0.189300,0.013095,1
2,feature_5,0.098765,0.009524,1
3,feature_4,0.078189,0.007540,1
4,feature_2,0.065844,0.005952,1
5,feature_7,0.045267,0.004365,1
6,feature_3,0.037037,-0.004365,1
7,feature_10,0.020576,0.000000,1
8,feature_1,0.000000,-0.002381,0
9,feature_11,0.000000,-0.008730,0


In [8]:
for slug in ["logistic_regression", "svm", "knn"]:
    print(f"\n{slug}")
    display(pd.read_csv(TASK4_DIR / slug / "feature_importance.csv").head(8))


logistic_regression


,feature,importance_mean,importance_std
0,feature_8,0.032143,0.049099
1,feature_5,0.000000,0.000000
2,feature_4,0.000000,0.000000
3,feature_6,0.000000,0.000000
4,feature_1,0.000000,0.000000
5,feature_7,0.000000,0.000000
6,feature_9,0.000000,0.000000
7,feature_2,-0.001190,0.040703



svm


,feature,importance_mean,importance_std
0,feature_1,0.0,0.0
1,feature_2,0.0,0.0
2,feature_3,0.0,0.0
3,feature_4,0.0,0.0
4,feature_5,0.0,0.0
5,feature_6,0.0,0.0
6,feature_7,0.0,0.0
7,feature_8,0.0,0.0



knn


,feature,importance_mean,importance_std
0,feature_8,0.096429,0.056205
1,feature_12,0.054762,0.040963
2,feature_5,0.028571,0.025085
3,feature_4,0.022619,0.017211
4,feature_2,0.019048,0.037721
5,feature_7,0.013095,0.017211
6,feature_3,0.010714,0.032143
7,feature_10,0.005952,0.029282


## Coeficientes de Regresion Logistica

Como complemento interpretable, se revisa la magnitud media de coeficientes de la regresion logistica optimizada. Este analisis no sustituye la permutation importance, pero ayuda a explicar la direccion e intensidad de la frontera lineal.

In [9]:
pd.read_csv(TASK4_DIR / "logistic_regression" / "logistic_coefficients.csv")

,feature,mean_abs_coefficient,max_abs_coefficient,signed_coefficient_sum
0,feature_7,1.093590,1.093590,1.093590
1,feature_8,0.564063,2.256254,-2.256254
2,feature_3,0.493578,1.240715,-1.000697
3,feature_2,0.475102,1.900407,1.900407
4,feature_12,0.449384,0.895820,0.443487
5,feature_4,0.394855,0.394855,0.394855
6,feature_11,0.357104,0.357104,0.357104
7,feature_13,0.237500,0.237500,0.237500
8,feature_10,0.208403,0.208403,-0.208403
9,feature_6,0.114433,0.114433,0.114433


## Cierre

El reporte consolidado queda en `reports/task4_optimizacion_parametros.md`. Los artefactos por modelo se guardan en `outputs/task4`, incluyendo grids completos, mejores parametros, metricas de test, matrices de confusion e importancia de variables.